In [14]:
import pandas as pd
import os
import glob

RAW_PATH = r"D:\CricMetric-AI\data\raw\odis"

files = glob.glob(os.path.join(RAW_PATH, "*.csv"))
print(f"Total match files found: {len(files)}")

Total match files found: 2544


In [ ]:
with open(files[0], 'r') as f:
    for i, line in enumerate(f):
        if i >= 74:
            print(f"Line {i}: {line.strip()}")
        if i > 85:
            break

Line 74: info,registry,people,TM Head,12b610c2
Line 75: info,registry,people,Umar Akmal,fd2bf2a0
Line 76: info,registry,people,Wahab Riaz,b3118300
Line 77: ball,1,0.1,Australia,DA Warner,TM Head,Mohammad Amir,0,0,,,,,,"","",0.1,,,,
Line 78: ball,1,0.2,Australia,DA Warner,TM Head,Mohammad Amir,0,0,,,,,,"","",0.2,,,,
Line 79: ball,1,0.3,Australia,DA Warner,TM Head,Mohammad Amir,0,0,,,,,,"","",0.3,,,,
Line 80: ball,1,0.4,Australia,DA Warner,TM Head,Mohammad Amir,0,0,,,,,,"","",0.4,,,,
Line 81: ball,1,0.5,Australia,DA Warner,TM Head,Mohammad Amir,0,1,1,,,,,"","",0.5,,,,
Line 82: ball,1,0.6,Australia,DA Warner,TM Head,Mohammad Amir,0,0,,,,,,"","",0.5,,,,
Line 83: ball,1,0.7,Australia,DA Warner,TM Head,Mohammad Amir,0,0,,,,,,"","",0.6,,,,
Line 84: ball,1,1.1,Australia,TM Head,DA Warner,Mohammad Hafeez,0,0,,,,,,"","",1.1,,,,
Line 85: ball,1,1.2,Australia,TM Head,DA Warner,Mohammad Hafeez,1,0,,,,,,"","",1.2,,,,
Line 86: ball,1,1.3,Australia,DA Warner,TM Head,Mohammad Hafeez,0,0,,,,,,"","",1.3,

In [ ]:
import pandas as pd
import os
import glob

RAW_PATH = r"D:\CricMetric-AI\data\raw\odis"

def parse_cricsheet_file(filepath):
    """
    Parse one Cricsheet CSV file into:
    - metadata dict (teams, date, venue etc)
    - ball-by-ball DataFrame
    """
    meta = {}
    ball_rows = []
    
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            
            # extract metadata from info lines
            if line.startswith('info,'):
                parts = line.split(',')
                if len(parts) >= 3:
                    key = parts[1]
                    value = ','.join(parts[2:])
                    # handle duplicate keys like team, player
                    if key in meta:
                        if not isinstance(meta[key], list):
                            meta[key] = [meta[key]]
                        meta[key].append(value)
                    else:
                        meta[key] = value
            
            # extract ball data
            elif line.startswith('ball,'):
                ball_rows.append(line.split(','))
    
    # column names from Cricsheet v1.8 format
    columns = [
        'type','innings','over','ball',
        'batting_team','batter','non_striker','bowler',
        'runs_off_bat','extras','wides','noballs',
        'byes','legbyes','penalty','wicket_type',
        'player_dismissed','other_wicket_type',
        'other_player_dismissed','super_over','replacement'
    ]
    
    if not ball_rows:
        return None, None
    
    # pad rows to match column count
    max_cols = len(columns)
    padded = [row + [''] * (max_cols - len(row)) for row in ball_rows]
    
    df = pd.DataFrame(padded, columns=columns)
    
    # add match metadata to every row
    match_id = os.path.basename(filepath).replace('.csv', '')
    df['match_id'] = match_id
    df['match_date'] = meta.get('date', '')
    df['teams'] = str(meta.get('team', ''))
    df['venue'] = meta.get('venue', '')
    df['event'] = meta.get('event', '')
    df['toss_winner'] = meta.get('toss_winner', '')
    df['city'] = meta.get('city', '')
    
    return df, meta

# test on one file first
sample_df, sample_meta = parse_cricsheet_file(files[0])
print("Metadata:", sample_meta)
print("\nShape:", sample_df.shape)
print("\nColumns:", sample_df.columns.tolist())
print("\nFirst 3 rows:")
sample_df.head(3)

Metadata: {'balls_per_over': '6', 'team': ['Australia', 'Pakistan'], 'team_type': 'international', 'gender': 'male', 'season': '2016/17', 'date': '2017/01/13', 'event': 'Pakistan in Australia ODI Series', 'match_id': '1000887', 'match_type': 'ODI', 'match_type_number': '3817', 'overs': '50', 'match_number': '1', 'venue': '"Brisbane Cricket Ground, Woolloongabba"', 'city': 'Brisbane', 'toss_winner': 'Australia', 'toss_decision': 'bat', 'player_of_match': 'MS Wade', 'umpire': ['MD Martell', 'C Shamshuddin'], 'reserve_umpire': 'SJ Nogajski', 'tv_umpire': 'CB Gaffaney', 'match_referee': 'JJ Crowe', 'target_overs': '2,50', 'target_runs': '2,269', 'winner': 'Australia', 'winner_runs': '92', 'player': ['Australia,DA Warner', 'Australia,TM Head', 'Australia,SPD Smith', 'Australia,CA Lynn', 'Australia,MR Marsh', 'Australia,GJ Maxwell', 'Australia,MS Wade', 'Australia,JP Faulkner', 'Australia,PJ Cummins', 'Australia,MA Starc', 'Australia,B Stanlake', 'Pakistan,Azhar Ali', 'Pakistan,Sharjeel Khan

,type,innings,over,ball,batting_team,batter,non_striker,bowler,runs_off_bat,extras,...,other_player_dismissed,super_over,replacement,match_id,match_date,teams,venue,event,toss_winner,city
0,ball,1,0.1,Australia,DA Warner,TM Head,Mohammad Amir,0,0,,...,,,,1000887,2017/01/13,"['Australia', 'Pakistan']","""Brisbane Cricket Ground, Woolloongabba""",Pakistan in Australia ODI Series,Australia,Brisbane
1,ball,1,0.2,Australia,DA Warner,TM Head,Mohammad Amir,0,0,,...,,,,1000887,2017/01/13,"['Australia', 'Pakistan']","""Brisbane Cricket Ground, Woolloongabba""",Pakistan in Australia ODI Series,Australia,Brisbane
2,ball,1,0.3,Australia,DA Warner,TM Head,Mohammad Amir,0,0,,...,,,,1000887,2017/01/13,"['Australia', 'Pakistan']","""Brisbane Cricket Ground, Woolloongabba""",Pakistan in Australia ODI Series,Australia,Brisbane


In [ ]:
import pandas as pd
import os
import glob

RAW_PATH = r"D:\CricMetric-AI\data\raw\odis"
files = glob.glob(os.path.join(RAW_PATH, "*.csv"))
print(f"Total match files found: {len(files)}")

def parse_cricsheet_file(filepath):
    meta = {}
    ball_rows = []
    
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            
            if line.startswith('info,'):
                # remove 'info,' prefix then split only on first comma
                content = line[5:]  # remove 'info,'
                parts = content.split(',', 1)
                if len(parts) >= 2:
                    key = parts[0].strip()
                    value = parts[1].strip().strip('"')
                    if key in meta:
                        if not isinstance(meta[key], list):
                            meta[key] = [meta[key]]
                        meta[key].append(value)
                    else:
                        meta[key] = value
            
            elif line.startswith('ball,'):
                # remove 'ball,' prefix then split
                content = line[5:]  # remove 'ball,'
                parts = content.split(',')
                ball_rows.append(parts)
    
    # correct columns without 'type' since we stripped 'ball,'
    columns = [
    'innings', 'over',
    'batting_team', 'batter', 'non_striker', 'bowler',
    'runs_off_bat', 'extras', 'wides', 'noballs',
    'byes', 'legbyes', 'penalty', 'wicket_type',
    'player_dismissed', 'other_wicket_type',
    'other_player_dismissed', 'running_total',
    'extra1', 'extra2', 'extra3'
]
    
    if not ball_rows:
        return None, None
    
    max_cols = len(columns)
    padded = [row + [''] * (max_cols - len(row)) for row in ball_rows]
    # trim if too many columns
    padded = [row[:max_cols] for row in padded]
    
    df = pd.DataFrame(padded, columns=columns)
    # add match info
    match_id = os.path.basename(filepath).replace('.csv', '')
    df['match_id'] = match_id
    df['match_date'] = meta.get('date', '')
    df['team1'] = meta.get('team', ['',''])[0] if isinstance(meta.get('team'), list) else meta.get('team', '')
    df['team2'] = meta.get('team', ['',''])[1] if isinstance(meta.get('team'), list) and len(meta.get('team')) > 1 else ''
    df['venue'] = meta.get('venue', '')
    df['event'] = meta.get('event', '')
    df['toss_winner'] = meta.get('toss_winner', '')
    df['city'] = meta.get('city', '')
    df['season'] = meta.get('season', '')
    
    return df, meta

# test on one file
sample_df, sample_meta = parse_cricsheet_file(files[0])

print("Match metadata:")
for k, v in sample_meta.items():
    if k not in ['registry', 'player']:
        print(f"  {k}: {v}")

print("\nShape:", sample_df.shape)
print("\nFirst 3 rows:")
sample_df

Total match files found: 2544
Match metadata:
  balls_per_over: 6
  team: ['Australia', 'Pakistan']
  team_type: international
  gender: male
  season: 2016/17
  date: 2017/01/13
  event: Pakistan in Australia ODI Series
  match_id: 1000887
  match_type: ODI
  match_type_number: 3817
  overs: 50
  match_number: 1
  venue: Brisbane Cricket Ground, Woolloongabba
  city: Brisbane
  toss_winner: Australia
  toss_decision: bat
  player_of_match: MS Wade
  umpire: ['MD Martell', 'C Shamshuddin']
  reserve_umpire: SJ Nogajski
  tv_umpire: CB Gaffaney
  match_referee: JJ Crowe
  target_overs: 2,50
  target_runs: 2,269
  winner: Australia
  winner_runs: 92

Shape: (571, 30)

First 3 rows:


,innings,over,batting_team,batter,non_striker,bowler,runs_off_bat,extras,wides,noballs,...,extra3,match_id,match_date,team1,team2,venue,event,toss_winner,city,season
0,1,0.1,Australia,DA Warner,TM Head,Mohammad Amir,0,0,,,...,,1000887,2017/01/13,Australia,Pakistan,"Brisbane Cricket Ground, Woolloongabba",Pakistan in Australia ODI Series,Australia,Brisbane,2016/17
1,1,0.2,Australia,DA Warner,TM Head,Mohammad Amir,0,0,,,...,,1000887,2017/01/13,Australia,Pakistan,"Brisbane Cricket Ground, Woolloongabba",Pakistan in Australia ODI Series,Australia,Brisbane,2016/17
2,1,0.3,Australia,DA Warner,TM Head,Mohammad Amir,0,0,,,...,,1000887,2017/01/13,Australia,Pakistan,"Brisbane Cricket Ground, Woolloongabba",Pakistan in Australia ODI Series,Australia,Brisbane,2016/17
3,1,0.4,Australia,DA Warner,TM Head,Mohammad Amir,0,0,,,...,,1000887,2017/01/13,Australia,Pakistan,"Brisbane Cricket Ground, Woolloongabba",Pakistan in Australia ODI Series,Australia,Brisbane,2016/17
4,1,0.5,Australia,DA Warner,TM Head,Mohammad Amir,0,1,1,,...,,1000887,2017/01/13,Australia,Pakistan,"Brisbane Cricket Ground, Woolloongabba",Pakistan in Australia ODI Series,Australia,Brisbane,2016/17
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
566,2,41.6,Pakistan,Wahab Riaz,Hasan Ali,JP Faulkner,1,0,,,...,,1000887,2017/01/13,Australia,Pakistan,"Brisbane Cricket Ground, Woolloongabba",Pakistan in Australia ODI Series,Australia,Brisbane,2016/17
567,2,42.1,Pakistan,Wahab Riaz,Hasan Ali,PJ Cummins,0,0,,,...,,1000887,2017/01/13,Australia,Pakistan,"Brisbane Cricket Ground, Woolloongabba",Pakistan in Australia ODI Series,Australia,Brisbane,2016/17
568,2,42.2,Pakistan,Wahab Riaz,Hasan Ali,PJ Cummins,1,0,,,...,,1000887,2017/01/13,Australia,Pakistan,"Brisbane Cricket Ground, Woolloongabba",Pakistan in Australia ODI Series,Australia,Brisbane,2016/17
569,2,42.3,Pakistan,Hasan Ali,Wahab Riaz,PJ Cummins,1,0,,,...,,1000887,2017/01/13,Australia,Pakistan,"Brisbane Cricket Ground, Woolloongabba",Pakistan in Australia ODI Series,Australia,Brisbane,2016/17


In [ ]:
sample_df['over_num'] = sample_df['over'].str.split('.').str[0].astype(int)
sample_df['ball_num'] = sample_df['over'].str.split('.').str[1].astype(int)

In [ ]:
sample_df.head(3)

,innings,over,batting_team,batter,non_striker,bowler,runs_off_bat,extras,wides,noballs,...,match_date,team1,team2,venue,event,toss_winner,city,season,over_num,ball_num
0,1,0.1,Australia,DA Warner,TM Head,Mohammad Amir,0,0,,,...,2017/01/13,Australia,Pakistan,"Brisbane Cricket Ground, Woolloongabba",Pakistan in Australia ODI Series,Australia,Brisbane,2016/17,0,1
1,1,0.2,Australia,DA Warner,TM Head,Mohammad Amir,0,0,,,...,2017/01/13,Australia,Pakistan,"Brisbane Cricket Ground, Woolloongabba",Pakistan in Australia ODI Series,Australia,Brisbane,2016/17,0,2
2,1,0.3,Australia,DA Warner,TM Head,Mohammad Amir,0,0,,,...,2017/01/13,Australia,Pakistan,"Brisbane Cricket Ground, Woolloongabba",Pakistan in Australia ODI Series,Australia,Brisbane,2016/17,0,3


In [ ]:
print(f"Files found: {len(files)}")
print(f"Sample shape: {sample_df.shape}")

Files found: 2544
Sample shape: (571, 32)


In [12]:
sample_df[['innings','over','over_num','ball_num','batting_team','batter','bowler','runs_off_bat']].head(5)

,innings,over,over_num,ball_num,batting_team,batter,bowler,runs_off_bat
0,1,0.1,0,1,Australia,DA Warner,Mohammad Amir,0
1,1,0.2,0,2,Australia,DA Warner,Mohammad Amir,0
2,1,0.3,0,3,Australia,DA Warner,Mohammad Amir,0
3,1,0.4,0,4,Australia,DA Warner,Mohammad Amir,0
4,1,0.5,0,5,Australia,DA Warner,Mohammad Amir,0


In [13]:
from tqdm import tqdm

all_dfs = []
failed_files = []

for filepath in tqdm(files, desc="Loading matches"):
    try:
        df, meta = parse_cricsheet_file(filepath)
        if df is not None:
            all_dfs.append(df)
    except Exception as e:
        failed_files.append((filepath, str(e)))

print(f"\nSuccessfully loaded: {len(all_dfs)} matches")
print(f"Failed: {len(failed_files)} matches")

Loading matches: 100%|██████████| 2544/2544 [01:51<00:00, 22.81it/s]


Successfully loaded: 2530 matches
Failed: 14 matches


In [15]:
print("Failed files and reasons:")
for filepath, error in failed_files:
    match_id = os.path.basename(filepath)
    print(f"\nMatch: {match_id}")
    print(f"Error: {error}")

Failed files and reasons:

Match: 1144528.csv
Error: Length of values (2) does not match length of index (619)

Match: 1263162.csv
Error: Length of values (3) does not match length of index (531)

Match: 1368002.csv
Error: Length of values (2) does not match length of index (392)

Match: 1388406.csv
Error: Length of values (2) does not match length of index (503)

Match: 247481.csv
Error: Length of values (2) does not match length of index (595)

Match: 256665.csv
Error: Length of values (2) does not match length of index (22)

Match: 473315.csv
Error: Length of values (2) does not match length of index (266)

Match: 597927.csv
Error: Length of values (2) does not match length of index (512)

Match: 660825.csv
Error: Length of values (2) does not match length of index (238)

Match: 66196.csv
Error: Length of values (2) does not match length of index (566)

Match: 66202.csv
Error: Length of values (2) does not match length of index (319)

Match: 66205.csv
Error: Length of values (2) doe

In [16]:
import pandas as pd
import os
import glob

RAW_PATH = r"D:\CricMetric-AI\data\raw\odis"
files = glob.glob(os.path.join(RAW_PATH, "*.csv"))
print(f"Total match files found: {len(files)}")

def parse_cricsheet_file(filepath):
    meta = {}
    ball_rows = []
    
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            
            if line.startswith('info,'):
                # remove 'info,' prefix then split only on first comma
                content = line[5:]  # remove 'info,'
                parts = content.split(',', 1)
                if len(parts) >= 2:
                    key = parts[0].strip()
                    value = parts[1].strip().strip('"')
                    if key in meta:
                        if not isinstance(meta[key], list):
                            meta[key] = [meta[key]]
                        meta[key].append(value)
                    else:
                        meta[key] = value
            
            elif line.startswith('ball,'):
                # remove 'ball,' prefix then split
                content = line[5:]  # remove 'ball,'
                parts = content.split(',')
                ball_rows.append(parts)
    
    # correct columns without 'type' since we stripped 'ball,'
    columns = [
    'innings', 'over',
    'batting_team', 'batter', 'non_striker', 'bowler',
    'runs_off_bat', 'extras', 'wides', 'noballs',
    'byes', 'legbyes', 'penalty', 'wicket_type',
    'player_dismissed', 'other_wicket_type',
    'other_player_dismissed', 'running_total',
    'extra1', 'extra2', 'extra3'
]
    
    if not ball_rows:
        return None, None
    
    max_cols = len(columns)
    padded = [row + [''] * (max_cols - len(row)) for row in ball_rows]
    # trim if too many columns
    padded = [row[:max_cols] for row in padded]
    
    df = pd.DataFrame(padded, columns=columns)
    # add match info
    match_id = os.path.basename(filepath).replace('.csv', '')
    df['match_id'] = match_id
    df['match_date'] = meta.get('date', '')
    teams = meta.get('team', [])
    if isinstance(teams, str):
      teams = [teams]
    df['team1'] = teams[0] if len(teams) > 0 else ''
    df['team2'] = teams[1] if len(teams) > 1 else ''
    df['venue'] = meta.get('venue', '')
    df['event'] = meta.get('event', '')
    df['toss_winner'] = meta.get('toss_winner', '')
    df['city'] = meta.get('city', '')
    df['season'] = meta.get('season', '')
    
    return df, meta

# test on one file
sample_df, sample_meta = parse_cricsheet_file(files[0])

print("Match metadata:")
for k, v in sample_meta.items():
    if k not in ['registry', 'player']:
        print(f"  {k}: {v}")

print("\nShape:", sample_df.shape)
print("\nFirst 3 rows:")
sample_df

Total match files found: 2544
Match metadata:
  balls_per_over: 6
  team: ['Australia', 'Pakistan']
  team_type: international
  gender: male
  season: 2016/17
  date: 2017/01/13
  event: Pakistan in Australia ODI Series
  match_id: 1000887
  match_type: ODI
  match_type_number: 3817
  overs: 50
  match_number: 1
  venue: Brisbane Cricket Ground, Woolloongabba
  city: Brisbane
  toss_winner: Australia
  toss_decision: bat
  player_of_match: MS Wade
  umpire: ['MD Martell', 'C Shamshuddin']
  reserve_umpire: SJ Nogajski
  tv_umpire: CB Gaffaney
  match_referee: JJ Crowe
  target_overs: 2,50
  target_runs: 2,269
  winner: Australia
  winner_runs: 92

Shape: (571, 30)

First 3 rows:


,innings,over,batting_team,batter,non_striker,bowler,runs_off_bat,extras,wides,noballs,...,extra3,match_id,match_date,team1,team2,venue,event,toss_winner,city,season
0,1,0.1,Australia,DA Warner,TM Head,Mohammad Amir,0,0,,,...,,1000887,2017/01/13,Australia,Pakistan,"Brisbane Cricket Ground, Woolloongabba",Pakistan in Australia ODI Series,Australia,Brisbane,2016/17
1,1,0.2,Australia,DA Warner,TM Head,Mohammad Amir,0,0,,,...,,1000887,2017/01/13,Australia,Pakistan,"Brisbane Cricket Ground, Woolloongabba",Pakistan in Australia ODI Series,Australia,Brisbane,2016/17
2,1,0.3,Australia,DA Warner,TM Head,Mohammad Amir,0,0,,,...,,1000887,2017/01/13,Australia,Pakistan,"Brisbane Cricket Ground, Woolloongabba",Pakistan in Australia ODI Series,Australia,Brisbane,2016/17
3,1,0.4,Australia,DA Warner,TM Head,Mohammad Amir,0,0,,,...,,1000887,2017/01/13,Australia,Pakistan,"Brisbane Cricket Ground, Woolloongabba",Pakistan in Australia ODI Series,Australia,Brisbane,2016/17
4,1,0.5,Australia,DA Warner,TM Head,Mohammad Amir,0,1,1,,...,,1000887,2017/01/13,Australia,Pakistan,"Brisbane Cricket Ground, Woolloongabba",Pakistan in Australia ODI Series,Australia,Brisbane,2016/17
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
566,2,41.6,Pakistan,Wahab Riaz,Hasan Ali,JP Faulkner,1,0,,,...,,1000887,2017/01/13,Australia,Pakistan,"Brisbane Cricket Ground, Woolloongabba",Pakistan in Australia ODI Series,Australia,Brisbane,2016/17
567,2,42.1,Pakistan,Wahab Riaz,Hasan Ali,PJ Cummins,0,0,,,...,,1000887,2017/01/13,Australia,Pakistan,"Brisbane Cricket Ground, Woolloongabba",Pakistan in Australia ODI Series,Australia,Brisbane,2016/17
568,2,42.2,Pakistan,Wahab Riaz,Hasan Ali,PJ Cummins,1,0,,,...,,1000887,2017/01/13,Australia,Pakistan,"Brisbane Cricket Ground, Woolloongabba",Pakistan in Australia ODI Series,Australia,Brisbane,2016/17
569,2,42.3,Pakistan,Hasan Ali,Wahab Riaz,PJ Cummins,1,0,,,...,,1000887,2017/01/13,Australia,Pakistan,"Brisbane Cricket Ground, Woolloongabba",Pakistan in Australia ODI Series,Australia,Brisbane,2016/17


In [17]:
from tqdm import tqdm

all_dfs = []
failed_files = []

for filepath in tqdm(files, desc="Loading matches"):
    try:
        df, meta = parse_cricsheet_file(filepath)
        if df is not None:
            all_dfs.append(df)
    except Exception as e:
        failed_files.append((filepath, str(e)))

print(f"\nSuccessfully loaded: {len(all_dfs)} matches")
print(f"Failed: {len(failed_files)} matches")

Loading matches: 100%|██████████| 2544/2544 [00:12<00:00, 203.46it/s]


Successfully loaded: 2530 matches
Failed: 14 matches


In [19]:
print("Failed files and reasons:")
for filepath, error in failed_files:
    match_id = os.path.basename(filepath)
    print(f"\nMatch: {match_id}")
    print(f"Filepath:{filepath}")
    print(f"Error: {error}")

Failed files and reasons:

Match: 1144528.csv
Filepath:D:\CricMetric-AI\data\raw\odis\1144528.csv
Error: Length of values (2) does not match length of index (619)

Match: 1263162.csv
Filepath:D:\CricMetric-AI\data\raw\odis\1263162.csv
Error: Length of values (3) does not match length of index (531)

Match: 1368002.csv
Filepath:D:\CricMetric-AI\data\raw\odis\1368002.csv
Error: Length of values (2) does not match length of index (392)

Match: 1388406.csv
Filepath:D:\CricMetric-AI\data\raw\odis\1388406.csv
Error: Length of values (2) does not match length of index (503)

Match: 247481.csv
Filepath:D:\CricMetric-AI\data\raw\odis\247481.csv
Error: Length of values (2) does not match length of index (595)

Match: 256665.csv
Filepath:D:\CricMetric-AI\data\raw\odis\256665.csv
Error: Length of values (2) does not match length of index (22)

Match: 473315.csv
Filepath:D:\CricMetric-AI\data\raw\odis\473315.csv
Error: Length of values (2) does not match length of index (266)

Match: 597927.csv
Fil

In [21]:
import pandas as pd
import os
import glob

RAW_PATH = r"D:\CricMetric-AI\data\raw\odis"
files = glob.glob(os.path.join(RAW_PATH, "*.csv"))
print(f"Total match files found: {len(files)}")

def parse_cricsheet_file(filepath):
    meta = {}
    ball_rows = []
    
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            
            if line.startswith('info,'):
                # remove 'info,' prefix then split only on first comma
                content = line[5:]  # remove 'info,'
                parts = content.split(',', 1)
                if len(parts) >= 2:
                    key = parts[0].strip()
                    value = parts[1].strip().strip('"')
                    if key in meta:
                        if not isinstance(meta[key], list):
                            meta[key] = [meta[key]]
                        meta[key].append(value)
                    else:
                        meta[key] = value
            
            elif line.startswith('ball,'):
                # remove 'ball,' prefix then split
                content = line[5:]  # remove 'ball,'
                parts = content.split(',')
                ball_rows.append(parts)
    
    # correct columns without 'type' since we stripped 'ball,'
    columns = [
    'innings', 'over',
    'batting_team', 'batter', 'non_striker', 'bowler',
    'runs_off_bat', 'extras', 'wides', 'noballs',
    'byes', 'legbyes', 'penalty', 'wicket_type',
    'player_dismissed', 'other_wicket_type',
    'other_player_dismissed', 'running_total',
    'extra1', 'extra2', 'extra3'
]
    
    if not ball_rows:
        return None, None
    
    max_cols = len(columns)
    padded = [row + [''] * (max_cols - len(row)) for row in ball_rows]
    # trim if too many columns
    padded = [row[:max_cols] for row in padded]
    
    df = pd.DataFrame(padded, columns=columns)
    # add match info
    match_id = os.path.basename(filepath).replace('.csv', '')
    df['match_id'] = match_id
    date = meta.get("date", "")
    if isinstance(date, list):
      date = date[0]
    df["match_date"] = date
    df['team1'] = meta.get('team', ['',''])[0] if isinstance(meta.get('team'), list) else meta.get('team', '')
    df['team2'] = meta.get('team', ['',''])[1] if isinstance(meta.get('team'), list) and len(meta.get('team')) > 1 else ''
    df['venue'] = meta.get('venue', '')
    df['event'] = meta.get('event', '')
    df['toss_winner'] = meta.get('toss_winner', '')
    df['city'] = meta.get('city', '')
    df['season'] = meta.get('season', '')
    
    return df, meta

# test on one file
sample_df, sample_meta = parse_cricsheet_file(files[0])

print("Match metadata:")
for k, v in sample_meta.items():
    if k not in ['registry', 'player']:
        print(f"  {k}: {v}")

print("\nShape:", sample_df.shape)
print("\nFirst 3 rows:")
sample_df

Total match files found: 2544
Match metadata:
  balls_per_over: 6
  team: ['Australia', 'Pakistan']
  team_type: international
  gender: male
  season: 2016/17
  date: 2017/01/13
  event: Pakistan in Australia ODI Series
  match_id: 1000887
  match_type: ODI
  match_type_number: 3817
  overs: 50
  match_number: 1
  venue: Brisbane Cricket Ground, Woolloongabba
  city: Brisbane
  toss_winner: Australia
  toss_decision: bat
  player_of_match: MS Wade
  umpire: ['MD Martell', 'C Shamshuddin']
  reserve_umpire: SJ Nogajski
  tv_umpire: CB Gaffaney
  match_referee: JJ Crowe
  target_overs: 2,50
  target_runs: 2,269
  winner: Australia
  winner_runs: 92

Shape: (571, 30)

First 3 rows:


,innings,over,batting_team,batter,non_striker,bowler,runs_off_bat,extras,wides,noballs,...,extra3,match_id,match_date,team1,team2,venue,event,toss_winner,city,season
0,1,0.1,Australia,DA Warner,TM Head,Mohammad Amir,0,0,,,...,,1000887,2017/01/13,Australia,Pakistan,"Brisbane Cricket Ground, Woolloongabba",Pakistan in Australia ODI Series,Australia,Brisbane,2016/17
1,1,0.2,Australia,DA Warner,TM Head,Mohammad Amir,0,0,,,...,,1000887,2017/01/13,Australia,Pakistan,"Brisbane Cricket Ground, Woolloongabba",Pakistan in Australia ODI Series,Australia,Brisbane,2016/17
2,1,0.3,Australia,DA Warner,TM Head,Mohammad Amir,0,0,,,...,,1000887,2017/01/13,Australia,Pakistan,"Brisbane Cricket Ground, Woolloongabba",Pakistan in Australia ODI Series,Australia,Brisbane,2016/17
3,1,0.4,Australia,DA Warner,TM Head,Mohammad Amir,0,0,,,...,,1000887,2017/01/13,Australia,Pakistan,"Brisbane Cricket Ground, Woolloongabba",Pakistan in Australia ODI Series,Australia,Brisbane,2016/17
4,1,0.5,Australia,DA Warner,TM Head,Mohammad Amir,0,1,1,,...,,1000887,2017/01/13,Australia,Pakistan,"Brisbane Cricket Ground, Woolloongabba",Pakistan in Australia ODI Series,Australia,Brisbane,2016/17
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
566,2,41.6,Pakistan,Wahab Riaz,Hasan Ali,JP Faulkner,1,0,,,...,,1000887,2017/01/13,Australia,Pakistan,"Brisbane Cricket Ground, Woolloongabba",Pakistan in Australia ODI Series,Australia,Brisbane,2016/17
567,2,42.1,Pakistan,Wahab Riaz,Hasan Ali,PJ Cummins,0,0,,,...,,1000887,2017/01/13,Australia,Pakistan,"Brisbane Cricket Ground, Woolloongabba",Pakistan in Australia ODI Series,Australia,Brisbane,2016/17
568,2,42.2,Pakistan,Wahab Riaz,Hasan Ali,PJ Cummins,1,0,,,...,,1000887,2017/01/13,Australia,Pakistan,"Brisbane Cricket Ground, Woolloongabba",Pakistan in Australia ODI Series,Australia,Brisbane,2016/17
569,2,42.3,Pakistan,Hasan Ali,Wahab Riaz,PJ Cummins,1,0,,,...,,1000887,2017/01/13,Australia,Pakistan,"Brisbane Cricket Ground, Woolloongabba",Pakistan in Australia ODI Series,Australia,Brisbane,2016/17


In [22]:
sample_df['over_num'] = sample_df['over'].str.split('.').str[0].astype(int)
sample_df['ball_num'] = sample_df['over'].str.split('.').str[1].astype(int)

In [23]:
from tqdm import tqdm

all_dfs = []
failed_files = []

for filepath in tqdm(files, desc="Loading matches"):
    try:
        df, meta = parse_cricsheet_file(filepath)
        if df is not None:
            all_dfs.append(df)
    except Exception as e:
        failed_files.append((filepath, str(e)))

print(f"\nSuccessfully loaded: {len(all_dfs)} matches")
print(f"Failed: {len(failed_files)} matches")

Loading matches: 100%|██████████| 2544/2544 [00:13<00:00, 188.93it/s]


Successfully loaded: 2544 matches
Failed: 0 matches
